# Notebook 3 — Generation Stage

For each retrieval run (query × method), sends the retrieved documents to the LLM
and extracts the citation order from the generated response.

**Input:** `data/retail/retail_geo_detailed_results.csv`

Columns: `query_id | query | doc_id | method | doc_text | rank | is_target`

**Output:** `data/retail/generation_results.parquet`

Columns: `query_id | original_query_id | method | query | target_new_position | num_docs | citation_order | boost_product_index | llm_response`

## Instructions
1. Run **Setup**
2. Run **Data Preparation**
3. Run **Helper Functions**
4. Run **Parameters**
5. Run **Generation**
6. Run **Inspect Results**
7. Run **Summary**

## Setup

In [1]:
import json
import os
import re
import sys
import time
from datetime import datetime

import pandas as pd
import numpy as np

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
sys.path.insert(0, os.path.join(project_root, "src"))

from llms import OpenAIHelper

config_path = os.path.join(project_root, "config.json")
with open(config_path, "r") as f:
    config = json.load(f)
os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]

data_dir = os.path.join(project_root, "data", "retail")

print("Setup complete.")
print(f"Project root: {project_root}")

Setup complete.
Project root: /Users/leonardrampf/Library/CloudStorage/OneDrive-Personal/Dokumente/Universität/Nova SBE/Work Project/geo-experiment


## Data Preparation

Load the retrieval CSV and extract `original_query_id` and `method` from `query_id`.
Example: `50881_Fluency(doc)` → `original_query_id=50881`, `method=Fluency(doc)`

In [2]:
retrieval_csv_path = os.path.join(data_dir, "retail_geo_detailed_results_v2.csv")
df = pd.read_csv(retrieval_csv_path)

# Extract original_query_id and method from query_id
df["original_query_id"] = df["query_id"].str.extract(r"^(\d+)_")
df["method"]            = df["query_id"].str.extract(r"^\d+_(.+)$")

# Flag runs that have no target document — these will be skipped
target_per_run = df.groupby("query_id")["is_target"].sum()
runs_with_target    = set(target_per_run[target_per_run == 1].index)
runs_without_target = set(target_per_run[target_per_run == 0].index)

# Save runs without target to a separate file for documentation
runs_without_target_path = os.path.join(data_dir, "runs_without_target.json")
with open(runs_without_target_path, "w") as f:
    json.dump(sorted(list(runs_without_target)), f, indent=4)

print(f"Total rows:              {len(df)}")
print(f"Unique runs (query_ids): {df['query_id'].nunique()}")
print(f"Unique original queries: {df['original_query_id'].nunique()}")
print(f"Unique methods:          {df['method'].nunique()}")
print(f"Runs with target:        {len(runs_with_target)}")
print(f"Runs without target:     {len(runs_without_target)} — saved to runs_without_target.json")
print()
print("Methods found:")
for m in sorted(df['method'].unique()):
    print(f"  {m}")

Total rows:              109535
Unique runs (query_ids): 10997
Unique original queries: 500
Unique methods:          22
Runs with target:        10940
Runs without target:     57 — saved to runs_without_target.json

Methods found:
  Authoritative(doc)
  CQ(doc)
  CQS(doc)
  CS(doc)
  Citations(doc)
  ContentImprovement(doc)
  FC(doc)
  FCQ(doc)
  FCQS(doc)
  FCS(doc)
  FQ(doc)
  FQS(doc)
  FS(doc)
  Fluency(doc)
  LLMstxt(doc)
  QS(doc)
  Quotes(doc)
  SimpleLanguage(doc)
  Statistics(doc)
  TechnicalTerms(doc)
  UniqueWords(doc)
  doc


## Helper Functions

In [3]:
DEVELOPER_PROMPT = """Your are a product recommender system for an online marketplace. Write an accurate and concise answer for the given user question, using _only_ the provided summarized web search results. The answer should be correct, high-quality, and written by an expert using an unbiased and journalistic tone. The answer should be informative, interesting, and engaging. The answer's logic and reasoning should be rigorous and defensible. Every sentence in the answer should be _immediately followed_ by an in-line citation to the search result(s). The cited search result(s) should fully support _all_ the information in the sentence. Search results need to be cited using [index]. When citing several search results, use [1][2][3] format rather than [1, 2, 3]. You can use multiple search results to respond comprehensively while avoiding irrelevant search results. Search results are not sorted."""
DOC_TYPE = "Product Description"


def build_context(group):
    """
    Takes all rows for one query_id, sorts by rank, re-numbers 1..N.
    Returns:
        docs: list of doc_text in retrieval order
        target_new_position: 0-based position of target doc after re-numbering
    """
    group = group.sort_values("rank").reset_index(drop=True)
    docs = group["doc_text"].tolist()

    # Find 0-based position of target doc
    target_new_position = None
    for i, row in group.iterrows():
        if row["is_target"] == 1:
            target_new_position = i  # 0-based
            break

    return docs, target_new_position


def build_user_prompt(query, docs):
    """
    Builds the user prompt for the LLM.
    docs: list of doc_text strings, numbered 1..N
    """
    sources = ""
    for i, doc_text in enumerate(docs, start=1):
        sources += f"[{i}] {DOC_TYPE}: {doc_text}\n\n"
    return f"Question: {query}\n\nSearch Results:\n{sources}"


def extract_citation_order(response_text):
    """
    Extracts citation indices in order of first appearance.
    0-based like Puerto: [1] in text -> index 0, [2] -> index 1, etc.
    e.g. "...product [2] is great [1][2]..." -> [1, 0]
    """
    citations = re.findall(r"\[(\d+)\]", response_text)
    seen = []
    for c in citations:
        idx = int(c) - 1  # 0-based
        if idx not in seen:
            seen.append(idx)
    return seen


print("Helper functions defined.")

Helper functions defined.


## Parameters

In [4]:
# LLM
LLM_NAME = "gpt-4o-mini-2024-07-18"

# Output
output_path = os.path.join(data_dir, "generation_results.parquet")

# All runs to process (only those with a target doc)
all_run_ids = sorted(runs_with_target)

# Check already done
if os.path.exists(output_path):
    df_done = pd.read_parquet(output_path)
    done_keys = set(df_done["query_id"].tolist())
    already_done = len(done_keys)
else:
    done_keys = set()
    already_done = 0

remaining = len(all_run_ids) - already_done

print(f"LLM:           {LLM_NAME}")
print(f"Total runs:    {len(all_run_ids)}")
print(f"Already done:  {already_done}")
print(f"Remaining:     {remaining}")
print(f"Estimated cost: ~${remaining * 0.0003:.2f}")

LLM:           gpt-4o-mini-2024-07-18
Total runs:    10940
Already done:  0
Remaining:     10940
Estimated cost: ~$3.28


## Generation

For each run (query_id):
1. Sort docs by rank and re-number 1..N
2. Build LLM prompt
3. Generate response with inline citations
4. Extract citation order
5. Save incrementally

Runs without a target doc are skipped.
Safe to interrupt and resume.

In [ ]:
llm = OpenAIHelper(LLM_NAME)

# Load existing results if any
if os.path.exists(output_path):
    results_df = pd.read_parquet(output_path)
    results = results_df.to_dict("records")
else:
    results = []

start_time = datetime.now()
print(f"Started at: {start_time.strftime('%H:%M:%S')}")
print(f"Total runs to process: {len(all_run_ids)}")
print()

for i, query_id in enumerate(all_run_ids):

    # Skip if already done
    if query_id in done_keys:
        continue

    # Get all rows for this run
    group = df[df["query_id"] == query_id]
    query_text         = group["query"].iloc[0]
    original_query_id  = group["original_query_id"].iloc[0]
    method             = group["method"].iloc[0]

    # Build context
    docs, target_new_position = build_context(group)

    if target_new_position is None:
        print(f"  [{i+1}] {query_id}: no target — skipping")
        continue

    # Build prompt
    user_prompt = build_user_prompt(query_text, docs)
    messages = [
        {"role": "system", "content": DEVELOPER_PROMPT},
        {"role": "user",   "content": user_prompt},
    ]

    try:
        response, _ = llm.generate(messages)
        response_text    = response.content
        citation_order   = extract_citation_order(response_text)

        results.append({
            "query_id":            query_id,
            "original_query_id":   original_query_id,
            "method":              method,
            "query":               query_text,
            "target_new_position": target_new_position,
            "num_docs":            len(docs),
            "citation_order":      citation_order,
            "llm_response":        response_text,
        })
        done_keys.add(query_id)
        pd.DataFrame(results).to_parquet(output_path, index=False)
        print(f"  [{i+1}/{len(all_run_ids)}] {query_id}: target at [{target_new_position}] | cited: {citation_order[:5]}")

    except Exception as e:
        print(f"  [{i+1}/{len(all_run_ids)}] {query_id}: ERROR — {e}")
        time.sleep(10)
        try:
            response, _    = llm.generate(messages)
            response_text  = response.content
            citation_order = extract_citation_order(response_text)
            results.append({
                "query_id":            query_id,
                "original_query_id":   original_query_id,
                "method":              method,
                "query":               query_text,
                "target_new_position": target_new_position,
                "num_docs":            len(docs),
                "citation_order":      citation_order,
                "llm_response":        response_text,
            })
            done_keys.add(query_id)
            pd.DataFrame(results).to_parquet(output_path, index=False)
            print(f"  [{i+1}/{len(all_run_ids)}] {query_id}: done (retry OK)")
        except Exception as e2:
            print(f"  [{i+1}/{len(all_run_ids)}] {query_id}: FAILED — {e2}")

end_time = datetime.now()
elapsed  = end_time - start_time
print(f"{'='*60}")
print(f"GENERATION COMPLETE")
print(f"Started:    {start_time.strftime('%H:%M:%S')}")
print(f"Finished:   {end_time.strftime('%H:%M:%S')}")
print(f"Total time: {str(elapsed).split('.')[0]}")
print(f"Results:    {len(results)} rows saved to {output_path}")

## Inspect Results

In [ ]:
df_results = pd.read_parquet(output_path)

INSPECT_QUERY_ID = df_results["query_id"].iloc[0]  # <- change to inspect other runs

row = df_results[df_results["query_id"] == INSPECT_QUERY_ID].iloc[0]

print(f"Run (query_id):    {row['query_id']}")
print(f"Original query:    {row['original_query_id']}")
print(f"Method:            {row['method']}")
print(f"Query:             {row['query']}")
print(f"Num docs:          {row['num_docs']}")
print(f"Target position:   {row['target_new_position']} (0-based)")
print(f"Citation order:    {row['citation_order']}")
print(f"Target cited:      {'YES' if row['target_new_position'] in row['citation_order'] else 'NO'}")
print(f"\n--- LLM Response ---")
print(row["llm_response"][:800])

## Summary

In [ ]:
df_results = pd.read_parquet(output_path)

print(f"Total runs done:  {len(df_results)}")
print(f"Total runs expected: {len(all_run_ids)}")
print(f"Remaining:        {len(all_run_ids) - len(df_results)}")
print()

# Citation rate per method
df_results["target_cited"] = df_results.apply(
    lambda row: 1 if row["target_new_position"] in row["citation_order"] else 0, axis=1
)

citation_rates = df_results.groupby("method")["target_cited"].mean().sort_values(ascending=False)
print("Citation rate per method (% runs where target was cited):")
for method, rate in citation_rates.items():
    bar = "\u2588" * int(rate * 20) + "\u2591" * (20 - int(rate * 20))
    print(f"  {method:30} [{bar}] {rate:.1%}")